In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Ptc.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 1000,2018-03-01 10:55:17,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 1000,2018-03-01 10:55:21,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,request for payment 1000,2018-03-01 11:34:16,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,2335.0
3,request for payment 1000,2018-03-01 11:34:23,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,7.0
4,request for payment 1000,2018-03-01 15:01:48,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,12445.0
5,request for payment 1000,2018-03-05 14:49:53,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,344885.0
6,request for payment 1000,2018-03-06 10:13:29,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request Payment,SYSTEM,UNDEFINED,69816.0
7,request for payment 1000,2018-03-08 17:31:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Payment Handled,SYSTEM,UNDEFINED,199051.0
8,request for payment 10043,2018-02-20 13:53:11,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
9,request for payment 10043,2018-02-20 13:53:14,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [6.00, 362277.00]                        61266.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [64.68, 1661.05]                         338.4588   quantile_derived    
case:Permit RequestedBudget    continuous     case     yes    [130.85, 4066.04]                        769

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Payment Handled', 'Request Payment'}]

In [14]:
engine.branching_sets

[{'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTED by PRE_APPROVER',
  'Permit REJECTED by SUPERVISOR',
  'Permit SUBMITTED by EMPLOYEE',
  'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR',
  'Request For Payment FINAL_APPROVED by SUPERVISOR',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by BUDGET OWNER',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED 

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Ptc-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 70592,4,1,0,0.395890,0.316780,0.475000,0.470000,0.009091,...,0.100789,0.000000,0.000000,0.000000,0.000000,0.0000,0.100789,0.100789,0.0,0.0
1,0,request for payment 29593,4,1,0,0.366524,0.358048,0.375000,0.490000,0.181818,...,0.341609,0.181818,0.141609,0.000000,0.283218,0.2000,0.000000,0.000000,0.0,0.0
2,0,request for payment 52969,6,1,0,0.401033,0.377067,0.425000,0.530000,0.400000,...,0.284461,0.400000,0.084461,0.000000,0.168923,0.2000,0.000000,0.000000,0.0,0.0
3,0,request for payment 55558,7,1,0,0.397709,0.345417,0.450000,0.550000,0.470588,...,0.419485,0.470588,0.119485,0.166667,0.072304,0.3000,0.000000,0.000000,0.0,0.0
4,0,request for payment 45517,8,1,2,0.362484,0.408300,0.316667,0.495000,0.526316,...,0.283334,0.526316,0.083334,0.166667,0.000002,0.2000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,18,request for payment 77073,8,1,2,0.412996,0.460993,0.365000,0.512500,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.0,0.0
162,18,request for payment 71978,8,1,2,0.394155,0.378310,0.410000,0.465625,0.015789,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.0,0.0
163,18,request for payment 82927,8,1,2,0.351685,0.228369,0.475000,0.462500,0.126316,...,0.112500,0.105263,0.050000,0.100000,0.000000,0.0625,0.000000,0.869437,0.0,1.0
164,18,request for payment 82310,8,1,2,0.296542,0.253085,0.340000,0.393750,0.000000,...,0.067115,0.000000,0.004615,0.000000,0.009230,0.0625,0.000000,0.000000,0.0,0.0


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()